# 6.1 算子调试与性能测量

前几章已经介绍了Vector、GEMM和融合算子的写法。开始开发自己的算子后，常见的问题是：输出与预期不一致，应该检查哪里？输出正确了，怎样取得它在NPU上的执行耗时？

本章介绍实际使用的调试和测量工具。6.2介绍算子输出与参考结果不一致时的排查方法，通过检查输入、中间结果和输出逐步定位问题；6.3介绍使用do_bench和msprof测量算子耗时的方法，以及如何读取和理解性能数据；6.4结合一个GEMM案例，练习定位并修复数值错误，再测量和解读算子耗时。

## 1. 先根据现象，确定下一步

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">观察到的现象</th>
<th style="text-align: left; vertical-align: top;">先做什么</th>
<th style="text-align: left; vertical-align: top;">阅读位置</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">程序编译失败或无法运行</td>
<td style="text-align: left; vertical-align: top;">读报错位置，检查输入约束、代码和环境</td>
<td style="text-align: left; vertical-align: top;"><a href="./06.02_debugging.ipynb">6.2第6部分</a></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">程序能运行，输出与参考结果不一致</td>
<td style="text-align: left; vertical-align: top;">选择中间量检查，找到差异首次出现的位置</td>
<td style="text-align: left; vertical-align: top;"><a href="./06.02_debugging.ipynb">6.2调试案例</a></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">输出正确，想知道kernel执行一次需要多久</td>
<td style="text-align: left; vertical-align: top;">用do_bench取得耗时，或用msprof采集并查看报告</td>
<td style="text-align: left; vertical-align: top;"><a href="./06.03_measure_and_tune.ipynb">6.3性能测量</a></td>
</tr>
</tbody>
</table>

排查时先提出一个可以检查的猜测，再寻找证据。例如，怀疑某块数据读错，就检查这次搬运的输入位置和局部结果。

**先保证结果正确：** 少计算一部分内容也可能让程序变快。数值错误尚未修复时，耗时更短不能说明优化有效。

## 2. 每节练习一种具体能力

- **[定位错误（6.2）](06.02_debugging.ipynb)：** 沿输入、搬运、计算和写回选择观察位置，核对索引，修复后比较完整输出。
- **[读取耗时（6.3）](06.03_measure_and_tune.ipynb)：** 使用do_bench和msprof，读懂返回值、单次调用记录和汇总数据。
- **[独立练习（6.4）](06.04_capstone.ipynb)：** 修复GEMM的读取错误，验证结果，再用两种工具测量耗时。

6.3沿用第4章的分块矩阵乘原理，使用一个较小的GEMM示例，逐行解释测量代码和采集命令，并展示950上的真实输出。读完后，可以把其中的kernel调用替换成自己的算子，再按同样的方式取得耗时。

<p class="course-note"><small>运行提示：线上环境不提供昇腾950（A5），可直接阅读页内代码和实测记录。需要运行时，请自行准备昇腾950环境。</small></p>

## 3. 先判断：现在应该做哪一步？

1. 一个GEMM输出有一整组列不对，修改流水参数后耗时变短。现在应该先检查什么？
2. 两个配置都通过了完整输出检查，但一个用较小的输入计时，另一个用较大的输入计时。能否根据耗时直接选择配置？

<details>
<summary>展开解析</summary>

1. 先检查错误列块使用的输入范围、计算与写回位置，找到数值错误的原因。修复后重新比较完整输出，再判断性能修改是否有效。
2. 不能。输入规模不同，计算量也不同。应先固定输入、数据类型和其他比较条件，再测量两个配置，才能判断这项配置修改的效果。

</details>

[章节导航](README.md) · [下一节](06.02_debugging.ipynb)